# APTOS 2019 Diabetic Retinopathy Grading: Colab T4 Training Notebook

This notebook is the GPU training companion for the production repo. It covers the complete pipeline requested:

1. Project setup and dependency install
2. Kaggle dataset download
3. EDA and data quality checks
4. Ben Graham / CLAHE preprocessing
5. Stratified 5-fold setup and dataloaders
6. Three model approaches: classifier, regression, ordinal/GeM
7. Training loop with AMP, QWK, checkpointing, W&B
8. Inference, TTA, fold/model ensembling, threshold optimization
9. XAI entry points: Grad-CAM and clinical explanation workflow
10. Packaging outputs for API, Streamlit, and deployment

Recommended Colab runtime: **T4 GPU**, High-RAM if available. On the menu: `Runtime -> Change runtime type -> T4 GPU`.

Clinical note: this project is for research/prototyping unless clinically validated under the appropriate regulatory pathway.

## 0. GPU Check

A T4 is enough for EfficientNet-B4/B5 at 512px and small-batch 768px training. ConvNeXt-Large and EfficientNet-B6 may require smaller batches, gradient accumulation, or Colab Pro/A100.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Bring The Repo Into Colab

Use one of these options:

- Option A: upload this project folder as a zip to Colab and unzip to `/content/diabetes`.
- Option B: push the repo to GitHub, then clone it here.
- Option C: store the repo in Google Drive and copy/sync it into `/content`.

The notebook expects the repo root to be `/content/diabetes`.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/diabetes")

# Option B example once you push the repo:
# !git clone https://github.com/YOUR_USERNAME/diabetes.git /content/diabetes

# Option A helper: if you upload diabetes.zip through the Colab sidebar, uncomment:
# !unzip -q /content/diabetes.zip -d /content

assert PROJECT_DIR.exists(), "Put the repo at /content/diabetes first."
os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

## 2. Install Dependencies

Colab already includes many scientific packages, but pin the core stack for reproducibility. `grad-cam` is the PyPI package name; the import is `pytorch_grad_cam`.

If install asks to restart runtime after dependency changes, restart and rerun from this point.

In [ ]:
!pip install -q --upgrade pip
!pip install -q   albumentations==1.4.21   timm==1.0.12   opencv-python-headless==4.10.0.84   imagehash==4.3.1   scikit-learn==1.6.0   scikit-image==0.24.0   scipy==1.14.1   pandas==2.2.3   matplotlib==3.9.3   PyYAML==6.0.2   tqdm==4.67.1   wandb==0.19.1   grad-cam==1.5.4   lime==0.2.0.1   shap==0.46.0   reportlab==4.2.5

!pip install -q pytest==8.3.4

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import numpy as np
import pandas as pd
import cv2
import timm
import torch
print("ready")

## 3. Google Drive + Kaggle Credentials

Use Drive for persistent checkpoints and processed images. Upload your `kaggle.json` from Kaggle Account settings.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, os

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/aptos_dr")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Upload kaggle.json when prompted.
if not Path("/root/.kaggle/kaggle.json").exists():
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
        shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

## 4. Download APTOS 2019 Dataset

Expected local structure after extraction:

```text
data/raw/aptos2019/train.csv
data/raw/aptos2019/test.csv
data/raw/aptos2019/train_images/*.png
data/raw/aptos2019/test_images/*.png
```

In [ ]:
RAW_DIR = PROJECT_DIR / "data/raw/aptos2019"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not (RAW_DIR / "train.csv").exists():
    !kaggle competitions download -c aptos2019-blindness-detection -p data/raw/aptos2019
    !unzip -q data/raw/aptos2019/aptos2019-blindness-detection.zip -d data/raw/aptos2019

!find data/raw/aptos2019 -maxdepth 2 -type f | head

## 5. Phase 1: EDA And Quality Audit

This creates class distribution plots, image-size distribution, samples per class, and quality CSVs for black/low-contrast/duplicate images.

In [ ]:
!python scripts/run_eda.py --config configs/config.yaml

from IPython.display import display, Image
for p in ["reports/figures/class_distribution.png", "reports/figures/image_size_distribution.png", "reports/figures/samples_per_class.png"]:
    print(p)
    display(Image(filename=p))

quality = pd.read_csv("reports/quality/image_quality_summary.csv")
display(quality)

### EDA Mistakes To Avoid

- Do not use random train/validation splits.
- Do not delete suspicious images blindly; inspect and document.
- Do not balance or augment validation data.
- Do not optimize anything on the test set.

## 6. Phase 2: Preprocessing

Default: Ben Graham preprocessing at 512px for T4 training. Later, try 768px for final runs.

512px: faster, good for iteration. 768px: better lesion detail, slower, smaller batch.

In [ ]:
# For first T4 run, use 512. Later repeat with --image-size 768.
!python scripts/preprocess_images.py --split train --method ben_graham --image-size 512
!python scripts/preprocess_images.py --split test --method ben_graham --image-size 512

# Optional alternative to compare:
# !python scripts/preprocess_images.py --split train --method clahe --image-size 512

## 7. Phase 3: Stratified 5-Fold Splits

APTOS is small and imbalanced. Use Stratified K-Fold, never a random split.

In [ ]:
!python scripts/create_folds.py --config configs/config.yaml --output data/interim/aptos2019_folds.csv
folds = pd.read_csv("data/interim/aptos2019_folds.csv")
display(pd.crosstab(folds.fold, folds.diagnosis))

## 8. Phase 4: Model Strategy

Run in this order:

1. **Approach A**: EfficientNet-B4 classifier. Fast sanity baseline.
2. **Approach B**: EfficientNet-B4 regression. Usually better aligned with QWK.
3. **Approach C**: EfficientNet-B5/B6 or ConvNeXt with ordinal thresholds + GeM. Best final candidate.

For a single free T4, start with regression B4 at 512px. A realistic first target is validation QWK 0.82-0.88. Pushing 0.90+ usually needs stronger preprocessing, threshold search, 5-fold ensembling, maybe external pretraining, and careful cleanup.

In [ ]:
# Patch config for a T4-friendly first run: regression EfficientNet-B4, 512px, small batch.
import yaml
from pathlib import Path

cfg_path = Path("configs/config.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["preprocessing"]["image_size"] = 512
cfg["model"]["architecture"] = "tf_efficientnet_b4_ns"
cfg["model"]["task"] = "regression"
cfg["model"]["dropout"] = 0.3
cfg["training"]["batch_size"] = 8
cfg["training"]["epochs"] = 10  # increase to 20-30 for real training
cfg["training"]["num_workers"] = 2
cfg["training"]["lr"] = 2e-4
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

## 9. Phase 5: Train One Fold

Use one fold first to validate the whole stack. Turn on W&B if you have an account; otherwise use `--no-wandb`.

In [ ]:
# Smoke/full first fold training. Remove --no-wandb after wandb login if desired.
!python scripts/train.py   --config configs/config.yaml   --fold-csv data/interim/aptos2019_folds.csv   --fold 0   --processed-dir data/processed/aptos2019/train/512px/ben_graham   --use-weighted-sampler   --no-wandb

## 10. Train All 5 Folds

This is the first serious ensemble. On free Colab, save checkpoints to Drive after each fold because sessions can disconnect.

In [ ]:
for fold in range(5):
    print("TRAIN FOLD", fold)
    !python scripts/train.py       --config configs/config.yaml       --fold-csv data/interim/aptos2019_folds.csv       --fold {fold}       --processed-dir data/processed/aptos2019/train/512px/ben_graham       --use-weighted-sampler       --no-wandb
    !cp -v models/checkpoints/*.pth /content/drive/MyDrive/aptos_dr/

## 11. Threshold Optimization

For regression and ordinal models, do not rely only on `round()`. Use validation predictions to optimize four thresholds for QWK.

The repo already provides `optimize_thresholds`; once validation prediction collection is added for every checkpoint, optimize thresholds on out-of-fold predictions only.

In [ ]:
from dr_grading.inference.thresholds import optimize_thresholds
from dr_grading.training.metrics import regression_to_classes, quadratic_weighted_kappa

# Example only. Replace with real OOF continuous predictions from all folds.
y_true = np.array([0,0,1,1,2,2,3,3,4,4])
y_pred = np.array([0.1,0.2,0.9,1.2,2.0,2.1,3.0,3.1,3.8,4.0])
th = optimize_thresholds(y_true, y_pred)
print("thresholds", th)
print("qwk", quadratic_weighted_kappa(y_true, regression_to_classes(y_pred, th)))

## 12. Phase 6: Inference And Submission

Use TTA and average continuous predictions from all fold checkpoints. Do not average rounded labels.

In [ ]:
# Example for one checkpoint. Repeat --checkpoint/--architecture/--task for every fold/model.
# Update checkpoint path names to match your actual saved files.
!python scripts/predict_submission.py   --test-csv data/raw/aptos2019/test.csv   --test-image-dir data/raw/aptos2019/test_images   --checkpoint models/checkpoints/tf_efficientnet_b4_ns_fold0.pth   --architecture tf_efficientnet_b4_ns   --task regression   --thresholds 0.5 1.5 2.5 3.5   --image-size 512   --output submission.csv

!head submission.csv

## 13. Phase 7: XAI Workflow

The repo currently starts Phase 7A with Grad-CAM. For Colab, use a trained checkpoint and generate heatmaps for clinical review. Later modules should add GradCAM++, EigenCAM, ScoreCAM, LayerCAM, AblationCAM, LIME, SHAP, landmark overlays, counterfactuals, and faithfulness metrics.

In [ ]:
# Minimal Grad-CAM usage skeleton after loading a trained model.
# This assumes the model/checkpoint path exists.
from PIL import Image as PILImage
from dr_grading.inference.predictor import CheckpointSpec, load_checkpoint_model
from dr_grading.data.transforms import build_valid_transforms
from dr_grading.data.preprocessing import PreprocessOptions, preprocess_fundus_image
from explainability.gradcam import GradCAMExplainer

# spec = CheckpointSpec(
#     path=Path("models/checkpoints/tf_efficientnet_b4_ns_fold0.pth"),
#     architecture="tf_efficientnet_b4_ns",
#     task="regression",
# )
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = load_checkpoint_model(spec, device)
# image_path = Path("data/raw/aptos2019/test_images/SOME_ID.png")
# original = np.asarray(PILImage.open(image_path).convert("RGB"))
# processed = preprocess_fundus_image(original, PreprocessOptions(image_size=512), method="ben_graham")
# tensor = build_valid_transforms(512)(image=processed)["image"]
# explainer = GradCAMExplainer(model, arch_name="tf_efficientnet_b4_ns", device=device)
# result = explainer.explain(tensor, original_image=original, preprocessed_image=processed, regression=True)
# display(result.overlay_original)
# display(result.overlay_preprocessed)

## 14. Optional Kaggle 2015 Pretraining

Use Kaggle 2015 for pretraining or warm-starting, not for APTOS validation. The validation folds and threshold search should remain APTOS-only.

Recommended strategy:

1. Train on 2015 at 512px with noisy labels for a few epochs.
2. Save backbone checkpoint.
3. Fine-tune on APTOS 5 folds.
4. Optimize thresholds on APTOS OOF predictions.

## 15. Production Export Checklist

After training:

- Copy checkpoints to Drive.
- Save thresholds JSON.
- Save config YAML used for training.
- Generate `submission.csv`.
- Use the best fold/model ensemble in FastAPI.
- Run XAI sanity checks: heatmaps should attend to lesions/vessels/macula, not borders or labels.

Most common production mistakes:

- Training with one preprocessing method and serving with another.
- Forgetting threshold optimization for regression/ordinal outputs.
- Saving checkpoints without architecture/task metadata.
- Reporting accuracy instead of QWK.
- Treating Grad-CAM as clinical proof instead of model-behavior evidence.

## 16. Suggested T4 Run Plan

Use this progression:

1. 512px EfficientNet-B4 regression, fold 0 only, 3-5 epochs: verify stack.
2. 512px EfficientNet-B4 regression, 5 folds, 20-30 epochs.
3. Optimize OOF thresholds.
4. 512px EfficientNet-B5 ordinal/GeM, 5 folds.
5. Optional 768px fine-tune if runtime/VRAM allows.
6. Ensemble B4 regression + B5 ordinal.
7. Run XAI review on high-confidence correct, low-confidence, and severe-grade samples.

A free T4 can get you a strong research model, but QWK 0.92+ is an aggressive target. Expect iteration on folds, thresholds, preprocessing, external pretraining, and ensemble weights.